# 02 — References, Copying, and Object Lifetime

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/02', {1: 'bf0104ecea739391', 2: '50dac10a9e938e12', 3: '8fc292185c3729d4', 4: 'bee5716de6e5dc44', 5: 'dbf50950d19a5443', 6: '493bb821a3dd1443', 7: '3838641995b69c13', 8: '35259c117f1ec20b', 9: 'edc3e99ac14c52e7'}, practice='advanced_01_internals_typing_architecture', independent='LRUCache')

## 1. Aliasing and copies

- Assignment never copies: `b = a` makes two names for one object. `is` checks identity, `==` checks value.
- `copy.copy(x)` is shallow (new outer container, same inner objects); `copy.deepcopy(x)` copies all the way down.
- Mutable default arguments are shared across calls: use `None` and create the list inside.

In [ ]:
import copy
a = [[1], [2]]
b = a
shallow = copy.copy(a)
deep = copy.deepcopy(a)
a[0].append(99)
print(b is a, shallow[0], deep[0])

**Exercise 1.** After `b = a; b.append(3)`, what is `a`?

In [ ]:
a = [1, 2]
b = a
b.append(3)
predicted = ___
check(1, predicted)

**Exercise 2.** A shallow copy shares the inner lists. Predict `shallow[0]` after `a[0].append(9)`.

In [ ]:
import copy
a = [[1], [2]]
shallow = copy.copy(a)
a[0].append(9)
predicted = ___
check(2, predicted)

**Exercise 3.** Copy all the way down so the inner list is independent.

In [ ]:
import copy
a = [[1], [2]]
c = ___   # a full, independent copy of a
a[0].append(9)
check(3, c[0])

**Exercise 4.** `x == y` and `x is y` for two separate equal lists?

In [ ]:
x, y = [1], [1]
predicted = ___   # (x == y, x is y)
check(4, predicted)

## 2. Lifetime and weak references

- CPython frees an object when its last reference goes; cycles are collected by `gc` later.
- `weakref.ref(obj)` points at an object without keeping it alive; calling it returns the object or `None` once it's gone.
- Caches that shouldn't keep things alive use `weakref.WeakValueDictionary`.

In [ ]:
import weakref, gc
class Node:
    pass
n = Node()
r = weakref.ref(n)
print(r() is n)
del n
gc.collect()
print(r())

**Exercise 5.** While the object is alive, does `r()` return it? Write `True` or `False`.

In [ ]:
import weakref
class Node: pass
n = Node()
r = weakref.ref(n)
predicted = ___   # r() is n
check(5, predicted)

**Exercise 6.** After the last strong reference is deleted, what does `r()` return?

In [ ]:
import weakref, gc
class Node: pass
n = Node()
r = weakref.ref(n)
del n
gc.collect()
predicted = ___
check(6, predicted)

## 3. Bounded caches

- Unbounded caches leak memory. Cap the size and evict the least recently used item (LRU).
- `collections.OrderedDict` keeps insertion order: `move_to_end(key)` marks it recent; `popitem(last=False)` removes the oldest.
- `functools.lru_cache(maxsize=128)` does this for function results.

In [ ]:
from collections import OrderedDict
cache = OrderedDict()
for key in ["a", "b", "c"]:
    cache[key] = key.upper()
cache.move_to_end("a")          # a is now most recent
cache.popitem(last=False)       # evicts b, the oldest
print(list(cache))

**Exercise 7.** Mark `"a"` as recently used, then list the keys.

In [ ]:
from collections import OrderedDict
d = OrderedDict.fromkeys(["a", "b", "c"])
d.move_to_end(___)
check(7, list(d))

**Exercise 8.** Evict the oldest key.

In [ ]:
from collections import OrderedDict
d = OrderedDict.fromkeys(["x", "y", "z"])
d.popitem(last=___)
check(8, list(d))

**Exercise 9.** A capacity-2 LRU: put a, put b, get a, put c. Which keys remain, oldest first?

In [ ]:
predicted = ___
check(9, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### Independent build — `LRUCache`

Implement LRUCache(capacity), rejecting nonpositive capacity with ValueError. get(key) returns the value or None on a miss. A hit or put(key, value) marks that key most recent. Evict the least recent on overflow. keys() lists keys least to most recent.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
class LRUCache:
    pass  # Implement the public interface in the ticket.

In [ ]:
check('LRUCache')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** copy.copy(list_of_dicts) copies…

   - **a)** Only the outer list; the dicts are shared
   - **b)** Nothing
   - **c)** The list and every dict

**2.** A WeakKeyDictionary entry disappears when…

   - **a)** You call clear() only
   - **b)** Its key object is garbage-collected
   - **c)** After 60 seconds

**3.** Reference cycles are cleaned up by…

   - **a)** The cyclic garbage collector
   - **b)** Never
   - **c)** Reference counting alone

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/02', {1: '2f7d90902338', 2: '193503e619f1', 3: '8e70bde26ebf'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [03 — Generics, TypeVars, Overloads, and Protocols](03_advanced_typing.ipynb)